In [0]:
# Step 2 (PII declaration and tagging) + secure-by-default ABAC for the restricted zones.
# Runs as the pseudonymization SP: it owns pii_raw / pii_vault (MANAGE) and is the only principal exempted.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
RAW, VAULT, GOV = f"{catalog}.pii_raw", f"{catalog}.pii_vault", f"{catalog}.pii_governance"

sp = spark.sql("SELECT current_user()").first()[0]
# Safety: only a service principal (application ID, no '@') may be put in the EXCEPT clause.
assert "@" not in sp, f"Refusing to exempt a human user from the PII policies: {sp}"
print(f"Exempt principal (pseudonymization SP): {sp}")

In [0]:
# Contract-driven tagging (preventive control). Data Classification writes the same class.* tags (detective control),
# so differences between the two show PII the contract missed.
rows = spark.sql(f"SELECT table_name, column_name, class_tag FROM {GOV}.pii_contract WHERE active AND class_tag IS NOT NULL").collect()
results = []
for r in rows:
    table = f"{RAW}.lfc_{r.table_name}"
    try:
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN `{r.column_name}` SET TAGS ('{r.class_tag}')")
        results.append((table, r.column_name, r.class_tag, "tagged"))
    except Exception as e:
        results.append((table, r.column_name, r.class_tag, f"FAILED: {str(e)[:200]}"))
display(spark.createDataFrame(results, "table STRING, column STRING, tag STRING, status STRING"))
failed = [x for x in results if x[3] != "tagged"]
assert not failed, f"{len(failed)} column tags failed (does the SP have ASSIGN on the class.* system tags?)"

In [0]:
# No control tag (account governed-tag quota is full), so the policies have no WHEN clause: they apply to every
# current and future table in the schema. Every column is masked (type-aware system mask) and every row is
# filtered for all account users, except the pseudonymization SP that runs the pipelines.
for schema, short in [(RAW, "raw"), (VAULT, "vault")]:
    spark.sql(f"""
    CREATE OR REPLACE POLICY pii_{short}_mask_all_columns
    ON SCHEMA {schema}
    COMMENT 'Secure by default: mask every column of every table in the restricted zone'
    COLUMN MASK system.data_classification.mask_value
    TO `account users` EXCEPT `{sp}`
    FOR TABLES
    MATCH COLUMNS TRUE AS c
    ON COLUMN c
    """)
    spark.sql(f"""
    CREATE OR REPLACE POLICY pii_{short}_deny_all_rows
    ON SCHEMA {schema}
    COMMENT 'Secure by default: no rows visible in the restricted zone'
    ROW FILTER {GOV}.pii_deny_all
    TO `account users` EXCEPT `{sp}`
    FOR TABLES
    """)
    display(spark.sql(f"SHOW POLICIES ON SCHEMA {schema}"))